# The ”News Feed” Extraction Pipeline

#### Convert a raw text feed of mixed items into a structured Excel report.

In [37]:
import sys
import pprint
sys.path.append('..')

from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model
from utils.json_utils import safe_parse_json, validate_json_schema, create_simple_schema, format_schema_for_prompt
import json
import warnings
from typing import Literal, Optional
import pandas as pd

from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv()

True

In [38]:
from pydantic import BaseModel, Field
from utils.json_utils import (
    format_pydantic_schema_for_prompt,
    parse_json_with_pydantic
)

class CrisisEvent(BaseModel):
    district : Literal["Colombo", "Gampaha", "Kandy", "Kalutara", "Galle", "Matara", "Ratnapura", "Nuwara Eliya", "Kegalle"] = Field(..., description="District where the event occurred")
    flood_level_meters : Optional[float] = Field(None, description="Flood level in meters/None if no flood")
    victim_count : int = Field(0, description="Number of victims, default to 0 if no victims")
    main_need : str = Field(..., description="Main need of the Feed")
    status : Literal["Critical", "Warning", "Stable"] = Field(..., description="Status of the Feed as Critical or Warning or Stable")

schema = format_pydantic_schema_for_prompt(CrisisEvent)
print(schema)

{
  "properties": {
    "district": {
      "description": "District where the event occurred",
      "enum": [
        "Colombo",
        "Gampaha",
        "Kandy",
        "Kalutara",
        "Galle",
        "Matara",
        "Ratnapura",
        "Nuwara Eliya",
        "Kegalle"
      ],
      "title": "District",
      "type": "string"
    },
    "flood_level_meters": {
      "anyOf": [
        {
          "type": "number"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Flood level in meters/None if no flood",
      "title": "Flood Level Meters"
    },
    "victim_count": {
      "default": 0,
      "description": "Number of victims, default to 0 if no victims",
      "title": "Victim Count",
      "type": "integer"
    },
    "main_need": {
      "description": "Main need of the Feed",
      "title": "Main Need",
      "type": "string"
    },
    "status": {
      "description": "Status of the Feed as Critical or Warn

In [39]:
model = pick_model('google', 'general')
client = LLMClient('google', model)

data_path = "../data/data/News Feed.txt"
with open(data_path, "r") as f:
    messages_list = [line.strip() for line in f if line.strip()]
 
print(f"Loaded {len(messages_list)} messages")

Loaded 30 messages


In [40]:
from IPython.core.display import JSON
valid_events = []
skipped = 0

for i, line in enumerate(messages_list, 1):
    prompt_text, spec = render(
        "json_extract.v1",
        schema=schema,
        text = line
    )

    response = client.json_chat(
        [{"role": "user", "content": prompt_text}],
        temperature=0.0
    )['text']

    success, parsed, parse_err = safe_parse_json(response)
    if not success:
        warnings.warn(f"[Line {i}] JSON parse failed: {parse_err} - skipping")
        skipped += 1
        continue

    try:
        json_str = json.dumps(parsed)
        event = CrisisEvent.model_validate_json(json_str)
        valid_events.append(event)

        print(f"[{i:02d}] {event.district} | {event.status}")

    except Exception as e:
        warnings.warn(f"[Line {i}] Validation failed:{e} - skipping")
        skipped += 1

print(f"\nValid events: {len(valid_events)}")
print(f"Skipped: {skipped}")


[01] Colombo | Critical
[02] Gampaha | Critical
[03] Kandy | Stable
[04] Kalutara | Critical
[05] Gampaha | Critical
[06] Colombo | Stable
[07] Matara | Stable
[08] Colombo | Critical
[09] Galle | Stable
[10] Gampaha | Warning
[11] Colombo | Critical
[12] Colombo | Stable
[13] Kandy | Critical
[14] Gampaha | Stable
[15] Nuwara Eliya | Warning
[16] Gampaha | Critical
[17] Ratnapura | Warning
[18] Colombo | Stable
[19] Kalutara | Critical
[20] Colombo | Critical
[21] Gampaha | Critical
[22] Kandy | Critical
[23] Gampaha | Critical
[24] Kalutara | Critical
[25] Matara | Warning
[26] Colombo | Critical
[27] Galle | Stable
[28] Gampaha | Critical
[29] Gampaha | Critical
[30] Kegalle | Critical

Valid events: 30
Skipped: 0


In [44]:
df = pd.DataFrame([e.model_dump() for e in valid_events])
output_path = "../outputs/flood_report.xlsx"
df.to_excel(output_path, index=False)

print(f"Saved {len(df)} events to {output_path}")
df.head(10)

Saved 30 events to ../outputs/flood_report.xlsx


,district,flood_level_meters,victim_count,main_need,status
0,Colombo,9.5,0,Flood relief,Critical
1,Gampaha,NaN,5,boat,Critical
2,Kandy,NaN,0,Traffic cleared,Stable
3,Kalutara,NaN,12,Rescue team,Critical
4,Gampaha,2.0,500,dry rations,Critical
5,Colombo,NaN,0,rescue,Stable
6,Matara,NaN,0,heavy rain,Stable
7,Colombo,1.5,1,Rescue,Critical
8,Galle,NaN,0,None,Stable
9,Gampaha,NaN,0,Dengue prevention,Warning
